In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_DIR = Path('../data')

In [ ]:
PANEL_COL = 'out.params.panel_constraint_overall.2023_nec_existing_dwelling_load_based'

baseline = pd.read_csv(DATA_DIR / 'IL_upgrade0.csv')

hp_upgrade = pd.read_csv(DATA_DIR / 'IL_upgrade4.csv', usecols=['bldg_id', PANEL_COL]).rename(
    columns={PANEL_COL: 'panel_constraint_hp'}
)

ev_upgrade = pd.read_csv(DATA_DIR / 'IL_upgrade23.csv', usecols=['bldg_id', PANEL_COL]).rename(
    columns={PANEL_COL: 'panel_constraint_ev'}
)

df = (
    baseline
    .merge(hp_upgrade, on='bldg_id', how='left')
    .merge(ev_upgrade, on='bldg_id', how='left')
)

df['panel_constraint_hp'] = df['panel_constraint_hp'].fillna('no upgrade applied')
df['panel_constraint_ev'] = df['panel_constraint_ev'].fillna('no upgrade applied')

print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
df.head()

In [ ]:
sf_types = ['Single-Family Attached', 'Single-Family Detached']

sf = df[df['in.geometry_building_type_acs'].isin(sf_types)]

display(
    sf.groupby(['panel_constraint_hp', 'panel_constraint_ev'])['weight']
    .sum()
    .reset_index()
    .rename(columns={'weight': 'weighted_units'})
    .sort_values('weighted_units', ascending=False)
)

In [ ]:
for label, col in [('Heat Pump', 'panel_constraint_hp'), ('EV', 'panel_constraint_ev')]:
    upgraded = sf[sf[col] != 'no upgrade applied']
    constrained = upgraded[upgraded[col] != 'No Constraint']
    pct = constrained['weight'].sum() / upgraded['weight'].sum() * 100
    print(f"{label}: {pct:.1f}% of upgraded homes have a panel constraint  (n={upgraded['weight'].sum():,.0f} weighted units)")

In [13]:
constrained_values = ['Capacity and Space Constrained', 'Capacity Constrained Only', 'Space Constrained Only']

constrained_either = sf[
    sf['panel_constraint_hp'].isin(constrained_values) |
    sf['panel_constraint_ev'].isin(constrained_values)
]

pct = constrained_either['weight'].sum() / sf['weight'].sum() * 100
print(f"{pct:.1f}% of single-family homes are panel constrained for either a heat pump or EV upgrade")

82.6% of single-family homes are panel constrained for either a heat pump or EV upgrade
